In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Section 1 — Timeseries Review

In [ ]:
# MAIN
from utils import (load_q_diepoldsau, load_q_gisingen,
                   load_ssc_diepoldsau, load_ssc_gisingen,
                   compute_monthly_mean, detrend_series)

# Load and resample to monthly means
q_diep   = compute_monthly_mean(load_q_diepoldsau())["Q"]
q_gis    = compute_monthly_mean(load_q_gisingen())["Q"]
ssc_diep = compute_monthly_mean(load_ssc_diepoldsau())["SSC"]
ssc_gis  = compute_monthly_mean(load_ssc_gisingen())["SSC"]

# Detrend: subtract significant linear trend (p<0.05) or mean
q_diep_dt,   t_q_diep   = detrend_series(q_diep)
q_gis_dt,    t_q_gis    = detrend_series(q_gis)
ssc_diep_dt, t_ssc_diep = detrend_series(ssc_diep)
ssc_gis_dt,  t_ssc_gis  = detrend_series(ssc_gis)

In [ ]:
# PLOT
from utils import plot_timeseries, plot_trend_fit

plot_timeseries({"Diepoldsau": q_diep, "Gisingen": q_gis},
                ylabel="Q [m\u00b3/s]", title="Monthly Mean Discharge")
plot_timeseries({"Diepoldsau": ssc_diep, "Gisingen": ssc_gis},
                ylabel="SSC [g/L]", title="Monthly Mean SSC")
plot_trend_fit(q_diep,   t_q_diep,   title="Q Diepoldsau — linear trend")
plot_trend_fit(q_gis,    t_q_gis,    title="Q Gisingen — linear trend")
plot_trend_fit(ssc_diep, t_ssc_diep, title="SSC Diepoldsau — linear trend")
plot_trend_fit(ssc_gis,  t_ssc_gis,  title="SSC Gisingen — linear trend")
plt.show()

In [ ]:
# PRINT
rows = [
    ("Q Diepoldsau",    t_q_diep),
    ("Q Gisingen",      t_q_gis),
    ("SSC Diepoldsau",  t_ssc_diep),
    ("SSC Gisingen",    t_ssc_gis),
]
df_trend = pd.DataFrame([{
    "Series":            name,
    "Slope (per month)": f"{t['slope']:.4e}",
    "p-value":           f"{t['p_value']:.4f}",
    "R\u00b2":           f"{t['r_squared']:.4f}",
    "Significant (5%)": t['significant'],
    "Detrending":        "trend subtracted" if t['significant'] else "mean subtracted"
} for name, t in rows])
print(df_trend.to_string(index=False))

**Comments:**

*(Discuss stationarity, observed trends, and whether any statistical trend in Q is due to a measuring error or a physical process.)*

# Section 2 — Timeseries Modelling

In [ ]:
# MAIN
from utils import (compute_acf_pacf, select_ar_order, select_arma_order,
                   fit_ar, fit_arma)

# Empirical ACF / PACF
corr_q_diep   = compute_acf_pacf(q_diep_dt,   lags=40)
corr_q_gis    = compute_acf_pacf(q_gis_dt,    lags=40)
corr_ssc_diep = compute_acf_pacf(ssc_diep_dt, lags=40)
corr_ssc_gis  = compute_acf_pacf(ssc_gis_dt,  lags=40)

# Select AR orders by AIC
p_ar_q_diep,   aic_ar_q_diep   = select_ar_order(q_diep_dt)
p_ar_q_gis,    aic_ar_q_gis    = select_ar_order(q_gis_dt)
p_ar_ssc_diep, aic_ar_ssc_diep = select_ar_order(ssc_diep_dt)
p_ar_ssc_gis,  aic_ar_ssc_gis  = select_ar_order(ssc_gis_dt)

# Select ARMA orders by AIC
p_arma_q_diep,   q_arma_q_diep,   _ = select_arma_order(q_diep_dt)
p_arma_q_gis,    q_arma_q_gis,    _ = select_arma_order(q_gis_dt)
p_arma_ssc_diep, q_arma_ssc_diep, _ = select_arma_order(ssc_diep_dt)
p_arma_ssc_gis,  q_arma_ssc_gis,  _ = select_arma_order(ssc_gis_dt)

# Fit models
ar_q_diep    = fit_ar(q_diep_dt,   p_ar_q_diep)
ar_q_gis     = fit_ar(q_gis_dt,    p_ar_q_gis)
ar_ssc_diep  = fit_ar(ssc_diep_dt, p_ar_ssc_diep)
ar_ssc_gis   = fit_ar(ssc_gis_dt,  p_ar_ssc_gis)

arma_q_diep   = fit_arma(q_diep_dt,   p_arma_q_diep,   q_arma_q_diep)
arma_q_gis    = fit_arma(q_gis_dt,    p_arma_q_gis,    q_arma_q_gis)
arma_ssc_diep = fit_arma(ssc_diep_dt, p_arma_ssc_diep, q_arma_ssc_diep)
arma_ssc_gis  = fit_arma(ssc_gis_dt,  p_arma_ssc_gis,  q_arma_ssc_gis)

In [ ]:
# PLOT
from utils import plot_acf_pacf

plot_acf_pacf(corr_q_diep,   title="Q Diepoldsau")
plot_acf_pacf(corr_q_gis,    title="Q Gisingen")
plot_acf_pacf(corr_ssc_diep, title="SSC Diepoldsau")
plot_acf_pacf(corr_ssc_gis,  title="SSC Gisingen")
plt.show()

In [ ]:
# PRINT
rows2 = [
    ("Q Diepoldsau",   p_ar_q_diep,   p_arma_q_diep,   q_arma_q_diep,
     round(ar_q_diep.aic, 1),   round(arma_q_diep.aic, 1)),
    ("Q Gisingen",     p_ar_q_gis,    p_arma_q_gis,    q_arma_q_gis,
     round(ar_q_gis.aic, 1),    round(arma_q_gis.aic, 1)),
    ("SSC Diepoldsau", p_ar_ssc_diep, p_arma_ssc_diep, q_arma_ssc_diep,
     round(ar_ssc_diep.aic, 1), round(arma_ssc_diep.aic, 1)),
    ("SSC Gisingen",   p_ar_ssc_gis,  p_arma_ssc_gis,  q_arma_ssc_gis,
     round(ar_ssc_gis.aic, 1),  round(arma_ssc_gis.aic, 1)),
]
df2 = pd.DataFrame(rows2, columns=["Series", "AR p", "ARMA p", "ARMA q",
                                    "AIC AR", "AIC ARMA"])
print(df2.to_string(index=False))

**Comments:**

*(Justify AR and ARMA order choices based on ACF/PACF. Discuss whether the same model order applies for Q at both locations, and same for C.)*

# Section 3 — Timeseries Application & Evaluation

In [ ]:
# MAIN
from utils import (plot_acf_comparison, plot_residual_acf,
                   portmanteau_test, normality_test)

# Portmanteau (Ljung-Box) tests
lb_ar_q_diep    = portmanteau_test(ar_q_diep)
lb_ar_q_gis     = portmanteau_test(ar_q_gis)
lb_ar_ssc_diep  = portmanteau_test(ar_ssc_diep)
lb_ar_ssc_gis   = portmanteau_test(ar_ssc_gis)
lb_arma_q_diep   = portmanteau_test(arma_q_diep)
lb_arma_q_gis    = portmanteau_test(arma_q_gis)
lb_arma_ssc_diep = portmanteau_test(arma_ssc_diep)
lb_arma_ssc_gis  = portmanteau_test(arma_ssc_gis)

# PPCC normality tests
ppcc_ar_q_diep,    pv_ar_q_diep,    _ = normality_test(ar_q_diep)
ppcc_ar_q_gis,     pv_ar_q_gis,     _ = normality_test(ar_q_gis)
ppcc_ar_ssc_diep,  pv_ar_ssc_diep,  _ = normality_test(ar_ssc_diep)
ppcc_ar_ssc_gis,   pv_ar_ssc_gis,   _ = normality_test(ar_ssc_gis)
ppcc_arma_q_diep,  pv_arma_q_diep,  _ = normality_test(arma_q_diep)
ppcc_arma_q_gis,   pv_arma_q_gis,   _ = normality_test(arma_q_gis)
ppcc_arma_ssc_diep,pv_arma_ssc_diep,_ = normality_test(arma_ssc_diep)
ppcc_arma_ssc_gis, pv_arma_ssc_gis, _ = normality_test(arma_ssc_gis)

In [ ]:
# PLOT

# Observed vs theoretical ACF
plot_acf_comparison(corr_q_diep,   ar_q_diep,    title="AR — Q Diepoldsau")
plot_acf_comparison(corr_q_diep,   arma_q_diep,  title="ARMA — Q Diepoldsau")
plot_acf_comparison(corr_q_gis,    ar_q_gis,     title="AR — Q Gisingen")
plot_acf_comparison(corr_q_gis,    arma_q_gis,   title="ARMA — Q Gisingen")
plot_acf_comparison(corr_ssc_diep, ar_ssc_diep,  title="AR — SSC Diepoldsau")
plot_acf_comparison(corr_ssc_diep, arma_ssc_diep,title="ARMA — SSC Diepoldsau")
plot_acf_comparison(corr_ssc_gis,  ar_ssc_gis,   title="AR — SSC Gisingen")
plot_acf_comparison(corr_ssc_gis,  arma_ssc_gis, title="ARMA — SSC Gisingen")

# Residual ACFs
plot_residual_acf(ar_q_diep,     title="AR — Q Diepoldsau")
plot_residual_acf(arma_q_diep,   title="ARMA — Q Diepoldsau")
plot_residual_acf(ar_q_gis,      title="AR — Q Gisingen")
plot_residual_acf(arma_q_gis,    title="ARMA — Q Gisingen")
plot_residual_acf(ar_ssc_diep,   title="AR — SSC Diepoldsau")
plot_residual_acf(arma_ssc_diep, title="ARMA — SSC Diepoldsau")
plot_residual_acf(ar_ssc_gis,    title="AR — SSC Gisingen")
plot_residual_acf(arma_ssc_gis,  title="ARMA — SSC Gisingen")

# Probability plots
for model, label in [
    (ar_q_diep,    "AR Q Diepoldsau"),  (arma_q_diep,   "ARMA Q Diepoldsau"),
    (ar_q_gis,     "AR Q Gisingen"),    (arma_q_gis,    "ARMA Q Gisingen"),
    (ar_ssc_diep,  "AR SSC Diepoldsau"),(arma_ssc_diep, "ARMA SSC Diepoldsau"),
    (ar_ssc_gis,   "AR SSC Gisingen"),  (arma_ssc_gis,  "ARMA SSC Gisingen"),
]:
    _, _, fig = normality_test(model)
    fig.axes[0].set_title(label)

plt.show()

In [ ]:
# PRINT
def lb_pass(df, alpha=0.05):
    return "PASS" if df["lb_pvalue"].min() > alpha else "FAIL"

rows3 = [
    ("Q Diepoldsau AR",     lb_pass(lb_ar_q_diep),    ppcc_ar_q_diep,    pv_ar_q_diep),
    ("Q Diepoldsau ARMA",   lb_pass(lb_arma_q_diep),  ppcc_arma_q_diep,  pv_arma_q_diep),
    ("Q Gisingen AR",       lb_pass(lb_ar_q_gis),     ppcc_ar_q_gis,     pv_ar_q_gis),
    ("Q Gisingen ARMA",     lb_pass(lb_arma_q_gis),   ppcc_arma_q_gis,   pv_arma_q_gis),
    ("SSC Diepoldsau AR",   lb_pass(lb_ar_ssc_diep),  ppcc_ar_ssc_diep,  pv_ar_ssc_diep),
    ("SSC Diepoldsau ARMA", lb_pass(lb_arma_ssc_diep),ppcc_arma_ssc_diep,pv_arma_ssc_diep),
    ("SSC Gisingen AR",     lb_pass(lb_ar_ssc_gis),   ppcc_ar_ssc_gis,   pv_ar_ssc_gis),
    ("SSC Gisingen ARMA",   lb_pass(lb_arma_ssc_gis), ppcc_arma_ssc_gis, pv_arma_ssc_gis),
]
df3 = pd.DataFrame(rows3,
    columns=["Model", "Portmanteau (5%)", "PPCC", "PPCC p-value"])
df3["PPCC"]         = df3["PPCC"].map("{:.4f}".format)
df3["PPCC p-value"] = df3["PPCC p-value"].map("{:.4f}".format)
print(df3.to_string(index=False))

**Comments:**

*(Discuss residual independence (Portmanteau) and normality (PPCC). Identify the most appropriate model (AR vs ARMA) for each series. Discuss parsimony and whether the chosen order should be adjusted.)*

# Section 4 — Ill to Rhein Relative Sediment Influence

In [ ]:
# MAIN
from utils import (simulate_arma, compute_sediment_mass,
                   monthly_yearly_yields, synthetic_mass_yields,
                   compare_statistics)

# Chosen models — update based on Section 3 analysis (AR vs ARMA)
chosen_q_diep   = arma_q_diep
chosen_q_gis    = arma_q_gis
chosen_ssc_diep = arma_ssc_diep
chosen_ssc_gis  = arma_ssc_gis

# Generate 10 synthetic series of 10 years (120 months)
sim_q_diep   = simulate_arma(chosen_q_diep,   n_months=120)
sim_q_gis    = simulate_arma(chosen_q_gis,    n_months=120)
sim_ssc_diep = simulate_arma(chosen_ssc_diep, n_months=120)
sim_ssc_gis  = simulate_arma(chosen_ssc_gis,  n_months=120)

# Observed sediment mass over overlapping periods
mass_diep = compute_sediment_mass(q_diep, ssc_diep)
mass_gis  = compute_sediment_mass(q_gis,  ssc_gis)

# Observed monthly / yearly yields
obs_monthly_diep, obs_yearly_diep = monthly_yearly_yields(mass_diep)
obs_monthly_gis,  obs_yearly_gis  = monthly_yearly_yields(mass_gis)

# Synthetic mass yields (add back original mean to restore physical scale)
syn_monthly_diep, syn_yearly_diep = synthetic_mass_yields(
    sim_q_diep, sim_ssc_diep, t_q_diep["original_mean"], t_ssc_diep["original_mean"])
syn_monthly_gis, syn_yearly_gis = synthetic_mass_yields(
    sim_q_gis, sim_ssc_gis, t_q_gis["original_mean"], t_ssc_gis["original_mean"])

# Statistical comparison of synthetic vs observed (normalised)
stats_q_diep   = compare_statistics(q_diep_dt,   sim_q_diep)
stats_q_gis    = compare_statistics(q_gis_dt,    sim_q_gis)
stats_ssc_diep = compare_statistics(ssc_diep_dt, sim_ssc_diep)
stats_ssc_gis  = compare_statistics(ssc_gis_dt,  sim_ssc_gis)

In [ ]:
# PLOT
from utils import plot_synthetic_vs_observed, plot_mass_yields

plot_synthetic_vs_observed(q_diep_dt,   sim_q_diep,   title="Synthetic vs Observed Q — Diepoldsau")
plot_synthetic_vs_observed(q_gis_dt,    sim_q_gis,    title="Synthetic vs Observed Q — Gisingen")
plot_synthetic_vs_observed(ssc_diep_dt, sim_ssc_diep, title="Synthetic vs Observed SSC — Diepoldsau")
plot_synthetic_vs_observed(ssc_gis_dt,  sim_ssc_gis,  title="Synthetic vs Observed SSC — Gisingen")

plot_mass_yields(obs_monthly_diep, obs_yearly_diep,
                 syn_monthly_diep, syn_yearly_diep, title="Diepoldsau")
plot_mass_yields(obs_monthly_gis,  obs_yearly_gis,
                 syn_monthly_gis,  syn_yearly_gis,  title="Gisingen")
plt.show()

In [ ]:
# PRINT
print("=== Statistical comparison: observed vs synthetic (normalised) ===\n")
for label, df in [("Q Diepoldsau",   stats_q_diep),
                  ("Q Gisingen",     stats_q_gis),
                  ("SSC Diepoldsau", stats_ssc_diep),
                  ("SSC Gisingen",   stats_ssc_gis)]:
    print(f"--- {label} ---")
    print(df.round(4).to_string())
    print()

obs_yearly_mean_diep = obs_yearly_diep.mean()
obs_yearly_mean_gis  = obs_yearly_gis.mean()
ill_fraction = obs_yearly_mean_gis / obs_yearly_mean_diep * 100

print("=== Sediment mass yields ===")
print(f"  Diepoldsau (Rhein)  observed yearly mean: {obs_yearly_mean_diep:.4f} kg/s")
print(f"  Gisingen   (Ill)    observed yearly mean: {obs_yearly_mean_gis:.4f} kg/s")
print(f"  Ill / Rhein contribution:                 {ill_fraction:.1f} %")
print()
print(f"  Diepoldsau synthetic yearly mean: {syn_yearly_diep:.4f} kg/s")
print(f"  Gisingen   synthetic yearly mean: {syn_yearly_gis:.4f} kg/s")

**Comments:**

*(Discuss whether synthetic series reproduce observed statistical characteristics. Quantify the Ill contribution to Rhein sediment load. Discuss implications of comparing non-synchronous time series.)*

# Section 5 — Independent Variables?

In [ ]:
# MAIN
from utils import compute_correlation

corr_diep = compute_correlation(q_diep, ssc_diep)
corr_gis  = compute_correlation(q_gis,  ssc_gis)

In [ ]:
# PLOT
from utils import plot_joint_distribution

plot_joint_distribution(q_diep, ssc_diep, title="Joint distribution Q–SSC — Diepoldsau")
plot_joint_distribution(q_gis,  ssc_gis,  title="Joint distribution Q–SSC — Gisingen")
plt.show()

In [ ]:
# PRINT
df5 = pd.DataFrame([
    {"Station": "Diepoldsau", **corr_diep},
    {"Station": "Gisingen",   **corr_gis},
])
df5 = df5.rename(columns={"pearson_r": "Pearson r", "pearson_p": "Pearson p",
                            "spearman_r": "Spearman r", "spearman_p": "Spearman p"})
for col in ["Pearson r", "Pearson p", "Spearman r", "Spearman p"]:
    df5[col] = df5[col].map("{:.4f}".format)
print(df5.to_string(index=False))

**Comments:**

*(Discuss the Q–SSC correlation at each station. Can Q and C be considered independent? Discuss the shortfalls of simulating dependent variables with independent models and how you would resolve them — e.g. copula models, joint bivariate simulation, or regression-based approaches.)*